# C-Top2Vec Preliminary Feasibility Test

**Paper:** Angelov & Inkpen (2024), *Topic Modeling: Contextual Token Embeddings Are All You Need*

## Objective

This notebook performs a preliminary feasibility assessment of Contextual-Top2Vec (C-Top2Vec) using the 20 Newsgroups dataset. The purpose is to verify that the published implementation can be executed successfully and that the principal outputs required for the COMP8240 reproduction project are accessible.

The full reproduction experiments will be conducted separately. This notebook initially uses a reduced subset of the original dataset to verify implementation feasibility within the available computational environment.

In [1]:
import sys
import numpy as np
import torch
import transformers

from top2vec import Top2Vec

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Python executable:", sys.executable)
print("MPS available:", torch.backends.mps.is_available())
print("Top2Vec import: SUCCESS")

/usr/local/Caskroom/miniforge/base/envs/ctop2vec-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.11.15
NumPy: 1.26.4
PyTorch: 2.2.2
Transformers: 4.41.2
Python executable: /usr/local/Caskroom/miniforge/base/envs/ctop2vec-env/bin/python
MPS available: True
Top2Vec import: SUCCESS


## 1. Load the 20 Newsgroups Dataset

The 20 Newsgroups dataset is used as the initial corpus for the feasibility assessment because it is one of the two benchmark datasets evaluated by Angelov and Inkpen (2024). The corpus contains 18,846 documents distributed across 20 predefined newsgroup categories.

For this implementation, the complete textual content is retained during retrieval. The ground-truth category labels are preserved for later evaluation but are not supplied to Contextual-Top2Vec during topic discovery.

In [2]:
from sklearn.datasets import fetch_20newsgroups

dataset = fetch_20newsgroups(
    subset="all",
    remove=()
)

print("Total documents:", len(dataset.data))
print("Number of categories:", len(dataset.target_names))

print("\nCategories:")
for category in dataset.target_names:
    print("-", category)

Total documents: 18846
Number of categories: 20

Categories:
- alt.atheism
- comp.graphics
- comp.os.ms-windows.misc
- comp.sys.ibm.pc.hardware
- comp.sys.mac.hardware
- comp.windows.x
- misc.forsale
- rec.autos
- rec.motorcycles
- rec.sport.baseball
- rec.sport.hockey
- sci.crypt
- sci.electronics
- sci.med
- sci.space
- soc.religion.christian
- talk.politics.guns
- talk.politics.mideast
- talk.politics.misc
- talk.religion.misc


## 2. Dataset Verification

The retrieved corpus is converted into a structured DataFrame containing the document text, numerical target label, and corresponding category name. Basic checks are performed to confirm the dataset dimensions, category distribution, and presence of missing or empty documents before preprocessing.

In [3]:
import pandas as pd

df = pd.DataFrame({
    "text": dataset.data,
    "target": dataset.target
})

df["category"] = df["target"].map(
    lambda x: dataset.target_names[x]
)

print("Dataset shape:", df.shape)
print("Number of categories:", df["category"].nunique())
print("Missing documents:", df["text"].isna().sum())
print("Empty documents:", (df["text"].str.strip() == "").sum())

print("\nDocuments per category:")
print(df["category"].value_counts().sort_index())

Dataset shape: (18846, 3)
Number of categories: 20
Missing documents: 0
Empty documents: 0

Documents per category:
category
alt.atheism                 799
comp.graphics               973
comp.os.ms-windows.misc     985
comp.sys.ibm.pc.hardware    982
comp.sys.mac.hardware       963
comp.windows.x              988
misc.forsale                975
rec.autos                   990
rec.motorcycles             996
rec.sport.baseball          994
rec.sport.hockey            999
sci.crypt                   991
sci.electronics             984
sci.med                     990
sci.space                   987
soc.religion.christian      997
talk.politics.guns          910
talk.politics.mideast       940
talk.politics.misc          775
talk.religion.misc          628
Name: count, dtype: int64


## 3. Data Preprocessing

Before constructing the feasibility sample, the corpus is checked for documents that may be unsuitable for modelling. Documents containing fewer than 35 characters are removed, following the minimum document-length criterion used in the original C-Top2Vec experiments.

No additional text cleaning, stemming, lemmatization, or stop-word removal is applied at this stage. The original document text is retained for subsequent contextual embedding and topic modelling.

In [4]:
# Calculate document lengths
df["length"] = df["text"].str.len()

# Apply minimum document-length threshold
df_clean = df[df["length"] >= 35].copy()
df_clean = df_clean.reset_index(drop=True)

print("Original documents:", len(df))
print("Documents after filtering:", len(df_clean))
print("Documents removed:", len(df) - len(df_clean))

print("\nMinimum document length:", df_clean["length"].min())
print("Maximum document length:", df_clean["length"].max())
print("Mean document length:", round(df_clean["length"].mean(), 2))

Original documents: 18846
Documents after filtering: 18846
Documents removed: 0

Minimum document length: 115
Maximum document length: 160616
Mean document length: 1902.53


## 4. Ground-Truth Labels

The 20 Newsgroups dataset provides predefined category labels for each document. These labels are retained separately for subsequent quantitative evaluation.

The labels are not provided to Contextual-Top2Vec during topic discovery. They serve only as external ground truth for evaluating how closely the discovered topic structure corresponds to the original document categories.

In [5]:
# Preserve the ground-truth labels for later evaluation
documents = df_clean["text"].tolist()
labels = df_clean["target"].to_numpy()

print("Number of documents:", len(documents))
print("Number of labels:", len(labels))
print("Number of unique labels:", len(np.unique(labels)))

Number of documents: 18846
Number of labels: 18846
Number of unique labels: 20


## 5. Feasibility Sample

Running Contextual-Top2Vec on the complete 20 Newsgroups corpus requires substantially greater computational resources than are available in the local environment. Therefore, a reduced stratified sample of 200 documents is used for the preliminary feasibility experiment.

Stratified sampling preserves representation across all 20 original Newsgroups categories. The category labels are used only to construct the sample and are not supplied to Contextual-Top2Vec during topic discovery.

This reduced experiment is intended to verify that the complete contextual topic-modelling pipeline can be executed successfully. Its results are not treated as a direct quantitative reproduction of the results reported by Angelov and Inkpen (2024).

In [6]:
from sklearn.model_selection import train_test_split

sample_df, _ = train_test_split(
    df_clean,
    train_size=200,
    stratify=df_clean["target"],
    random_state=42
)

sample_df = sample_df.reset_index(drop=True)

sample_documents = sample_df["text"].tolist()
sample_labels = sample_df["target"].to_numpy()

print("Sample size:", len(sample_documents))
print("Number of categories:", sample_df["target"].nunique())

print("\nDocuments per category:")
print(sample_df["category"].value_counts().sort_index())

Sample size: 200
Number of categories: 20

Documents per category:
category
alt.atheism                  8
comp.graphics               10
comp.os.ms-windows.misc     10
comp.sys.ibm.pc.hardware    10
comp.sys.mac.hardware       10
comp.windows.x              10
misc.forsale                10
rec.autos                   11
rec.motorcycles             11
rec.sport.baseball          11
rec.sport.hockey            11
sci.crypt                   11
sci.electronics             10
sci.med                     11
sci.space                   10
soc.religion.christian      11
talk.politics.guns          10
talk.politics.mideast       10
talk.politics.misc           8
talk.religion.misc           7
Name: count, dtype: int64


## 6. Contextual-Top2Vec Configuration

The feasibility experiment applies the Contextual-Top2Vec implementation using the MPNet sentence-transformer embedding model. The configuration follows the general experimental setup described by Angelov and Inkpen (2024), with adjustments to the clustering parameters to support execution on the substantially smaller 200-document sample.

The reduced configuration should therefore be interpreted as a computational feasibility test rather than an exact reproduction of the original 20 Newsgroups experiment. Its purpose is to verify that contextual token embeddings, dimensionality reduction, density-based clustering, topic discovery, and document-level topic segmentation can be generated successfully in the local environment.

The following parameters are used for the feasibility experiment.

In [7]:
from top2vec import Top2Vec
import time

umap_args = {
    "n_neighbors": 15,
    "n_components": 5
}

hdbscan_args = {
    "min_cluster_size": 5
}

print("Feasibility configuration")
print("-------------------------")
print("Documents:", len(sample_documents))
print("Embedding model: all-mpnet-base-v2")
print("Contextual Top2Vec: True")
print("Smoothing window: 3")
print("Minimum word count: 5")
print("UMAP n_neighbors:", umap_args["n_neighbors"])
print("UMAP n_components:", umap_args["n_components"])
print("HDBSCAN min_cluster_size:", hdbscan_args["min_cluster_size"])

Feasibility configuration
-------------------------
Documents: 200
Embedding model: all-mpnet-base-v2
Contextual Top2Vec: True
Smoothing window: 3
Minimum word count: 5
UMAP n_neighbors: 15
UMAP n_components: 5
HDBSCAN min_cluster_size: 5


## 7. Feasibility Model Training

Contextual-Top2Vec is now applied to the 200-document stratified sample. The purpose of this run is to confirm that the complete modelling pipeline can be executed successfully and that its principal outputs can be accessed programmatically.

The resulting topic count and topic assignments are treated as preliminary feasibility outputs only. Because the corpus size and clustering thresholds differ substantially from the full experimental configuration reported by Angelov and Inkpen (2024), the numerical results from this run are not used as direct reproduction results.

In [39]:
start_time = time.time()

model = Top2Vec(
    documents=sample_documents,
    contextual_top2vec=True,
    c_top2vec_smoothing_window=3,
    embedding_model="all-mpnet-base-v2",
    embedding_batch_size=4,
    min_count=5,
    umap_args=umap_args,
    hdbscan_args=hdbscan_args,
    workers=4,
    verbose=True
)

elapsed = time.time() - start_time

print("\nTraining completed.")
print(f"Runtime: {elapsed / 60:.2f} minutes")
print("Topics discovered:", model.get_num_topics())

2026-08-22 01:19:13,406 - top2vec - INFO - Pre-processing documents for training
/usr/local/Caskroom/miniforge/base/envs/ctop2vec-env/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2026-08-22 01:19:13,665 - top2vec - INFO - Creating vocabulary embedding
Embedding vocabulary: 100%|████████████████████████████████████████████████████████████| 34/34 [01:52<00:00,  3.30s/it]
2026-08-22 01:21:08,987 - top2vec - INFO - Create contextualized document embeddings
Embedding documents: 100%|███████████████████████████████████████████████████████████████| 7/7 [00:20<00:00,  2.88s/it]
200it [00:00, 1820.00it/s]
2026-08-22 01:21:32,663 - top2vec - INFO - Creating lower dimension embedding of documents
2026-08-22 01:21:36,408 - top2vec - INFO - Finding dense areas of documents
2026-08-22 01:21:36,484 - top2vec - INFO - Finding topics
Calculating document topic distribution


Training completed.
Runtime: 2.43 minutes
Topics discovered: 163


## 8. Saving the Feasibility Model

The successfully trained feasibility model is saved locally so that the resulting topic structure and document-level outputs can be reloaded without repeating the computationally expensive embedding and clustering stages.

Saving the trained model also improves the reproducibility of the subsequent inspection and evaluation steps by ensuring that the same feasibility result is used throughout the remainder of the notebook.

In [40]:
import os

os.makedirs("models", exist_ok=True)

model.save("models/ctop2vec_200")

print("Model saved successfully.")
print("Files in models directory:")
print(os.listdir("models"))

Model saved successfully.
Files in models directory:
['ctop2vec_200']


In [11]:
from top2vec import Top2Vec

model = Top2Vec.load("models/ctop2vec_200")

print("Model reload: SUCCESS")
print("Topics after reload:", model.get_num_topics())

Model reload: SUCCESS
Topics after reload: 163


## 8.1 Preliminary Topic Assignment Evaluation

To obtain an initial quantitative indication of topic-assignment performance, the document-topic distribution and document-topic relevance scores from the saved 200-document feasibility model are combined using element-wise multiplication, following the evaluation procedure described by Angelov and Inkpen (2024). Each document is assigned to the topic with the highest combined score, and the resulting assignments are compared with the original 20 Newsgroups category labels using ARI and AMI.

Because this feasibility experiment uses only 200 documents and modified clustering parameters, the results are treated as preliminary and are not considered an exact reproduction of the original full-dataset experiment.

In [9]:
import numpy as np
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score

# Retrieve C-Top2Vec document-level outputs
topic_distribution = model.get_document_topic_distribution()
topic_relevance = model.get_document_topic_relevance()

print("Topic distribution shape:", topic_distribution.shape)
print("Topic relevance shape:", topic_relevance.shape)

# Combine probability and relevance following Angelov & Inkpen (2024)
combined_scores = topic_distribution * topic_relevance

# Assign each document to the topic with the highest combined score
predicted_topics = np.argmax(combined_scores, axis=1)

# Ground-truth labels from our stratified 20 Newsgroups sample
true_labels = sample_labels

# Calculate clustering agreement
ari = adjusted_rand_score(true_labels, predicted_topics)
ami = adjusted_mutual_info_score(true_labels, predicted_topics)

print("\nPilot evaluation results")
print("------------------------")
print("ARI:", round(ari, 4))
print("AMI:", round(ami, 4))
print("Number of predicted topics:", len(np.unique(predicted_topics)))
print("Number of true categories:", len(np.unique(true_labels)))

Topic distribution shape: (200, 163)
Topic relevance shape: (200, 163)

Pilot evaluation results
------------------------
ARI: 0.0567
AMI: 0.1329
Number of predicted topics: 163
Number of true categories: 20


In [10]:
import pandas as pd

comparison = pd.DataFrame({
    "Metric": ["ARI", "AMI"],
    "Pilot_200_docs": [ari, ami],
    "Paper_20_Newsgroups": [0.442, 0.582]
})

comparison

,Metric,Pilot_200_docs,Paper_20_Newsgroups
0,ARI,0.056690,0.442
1,AMI,0.132895,0.582


### Preliminary Interpretation

The reduced feasibility run successfully reproduced the C-Top2Vec pipeline, but the resulting topic structure was highly fragmented. On the 200-document sample, the model produced 163 topics compared with 20 reference categories. This resulted in relatively low agreement with the ground-truth labels, with an ARI of 0.0567 and an AMI of 0.1329.

These results should be interpreted as preliminary feasibility findings rather than an exact reproduction of the original study, because the pilot used a much smaller sample and modified clustering parameters to accommodate local computational constraints.

## 9. Inspection of Discovered Topics

Following successful model training, the discovered topics are inspected qualitatively to confirm that the feasibility implementation produces interpretable thematic groupings.

For each topic, Contextual-Top2Vec returns a ranked list of representative terms. The inspection below displays the first ten discovered topics together with their highest-ranked terms. This qualitative assessment is intended only to verify that the model generates coherent topic descriptions; formal quantitative evaluation is conducted separately.

In [42]:
topic_words, word_scores, topic_nums = model.get_topics()

print("Total topics:", len(topic_nums))

for i in range(min(10, len(topic_nums))):
    print(f"\nTopic {topic_nums[i]}")
    print(", ".join(topic_words[i][:10]))

Total topics: 163

Topic 0
clemens, morris, bonds, playoff, batting, hayes, statistics, sort, baseball, evidence

Topic 1
clemens, batting, dh, rest, hayes, playoff, bonds, baseball, olson, season

Topic 2
bonds, williams, clemens, batting, dh, second, behind, order, reasons, sort

Topic 3
ducati, bike, rider, engine, max, vax, driving, technical, mode, equipment

Topic 4
marketing, companies, corporation, novell, mailing, dos, corp, company, manufacturers, products

Topic 5
nhl, ahl, hockey, defense, playoff, division, henrik, stein, statistics, goal

Topic 6
atheists, morality, physics, inherent, theory, opinion, nuclear, society, evidence, sciences

Topic 7
centaur, cwru, karner, ron, turpin, enzo, rd, roger, apr, stratus

Topic 8
sciences, opinion, opinions, physics, engineering, evidence, effort, research, statistics, behavior

Topic 9
rates, cards, price, card, au, apr, colour, blue, lower, graphics


## 10. Document-Level Contextual Outputs

A key feature of Contextual-Top2Vec is its ability to represent multiple topical segments within an individual document. To examine this functionality, the document-level topic distributions, topic relevance scores, tokenized documents, and token-level topic assignments are retrieved from the trained model.

These outputs provide the basis for subsequent inspection of within-document topic assignments and contextual topic segmentation.

In [43]:
topic_distribution = model.get_document_topic_distribution()
topic_relevance = model.get_document_topic_relevance()
tokens = model.get_document_tokens()
token_assignments = model.get_document_token_topic_assignment()

print("Topic distribution shape:", topic_distribution.shape)
print("Topic relevance shape:", topic_relevance.shape)
print("Number of tokenized documents:", len(tokens))
print("Number of token assignment records:", len(token_assignments))

Topic distribution shape: (200, 163)
Topic relevance shape: (200, 163)
Number of tokenized documents: 200
Number of token assignment records: 200


## 11. Distribution of Topic Assignments Across Documents

To examine the document-level behaviour of the feasibility model, the number of non-zero topic assignments is calculated for each document. This provides an initial indication of whether Contextual-Top2Vec identifies multiple topical regions within individual documents.

The analysis also identifies documents receiving the largest number of topic assignments for subsequent qualitative inspection. At this stage, the number of assigned topics is descriptive only and should not be interpreted as evidence that the identified topical segments are necessarily accurate.

In [44]:
import numpy as np

# Count the number of non-zero topic assignments per document
nonzero_topic_counts = (topic_distribution > 0).sum(axis=1)

print("Minimum topics in a document:", nonzero_topic_counts.min())
print("Maximum topics in a document:", nonzero_topic_counts.max())
print("Mean topics per document:", round(nonzero_topic_counts.mean(), 2))

# Identify documents with the most assigned topics
candidate_docs = np.argsort(nonzero_topic_counts)[::-1][:10]

print("\nDocuments with the most assigned topics:")

for doc_id in candidate_docs:
    print(
        f"Document {doc_id}: "
        f"{nonzero_topic_counts[doc_id]} topics | "
        f"Category: {sample_df.iloc[doc_id]['category']}"
    )

Minimum topics in a document: 1
Maximum topics in a document: 8
Mean topics per document: 1.1

Documents with the most assigned topics:
Document 127: 8 topics | Category: sci.electronics
Document 180: 7 topics | Category: sci.med
Document 97: 2 topics | Category: misc.forsale
Document 103: 2 topics | Category: comp.graphics
Document 111: 2 topics | Category: sci.electronics
Document 88: 2 topics | Category: comp.windows.x
Document 173: 2 topics | Category: comp.sys.ibm.pc.hardware
Document 50: 2 topics | Category: sci.electronics
Document 197: 2 topics | Category: sci.space
Document 61: 1 topics | Category: sci.crypt


## 12. Qualitative Inspection of a Multi-Topic Document

To examine the contextual topic assignments in greater detail, the document receiving the largest number of non-zero topic assignments is selected for qualitative inspection. Document 127 was assigned eight topics, the highest number observed within the 200-document feasibility sample.

The original document text and its assigned topics are examined together to assess whether the multiple topic assignments correspond to meaningful content within the document. This inspection is exploratory and does not constitute a formal evaluation of segmentation accuracy.

In [46]:
# Retrieve topic words once and construct a lookup table
all_topic_words, all_word_scores, all_topic_nums = model.get_topics()

topic_word_lookup = {
    topic_num: words
    for topic_num, words in zip(all_topic_nums, all_topic_words)
}

# Select the document with the highest number of assigned topics
doc_id = int(np.argmax(nonzero_topic_counts))

print("DOCUMENT", doc_id)
print("=" * 80)

print("\nOriginal 20 Newsgroups category:")
print(sample_df.iloc[doc_id]["category"])

print("\nOriginal document:")
print("-" * 80)
print(sample_documents[doc_id])

print("\n" + "=" * 80)
print("ASSIGNED TOPICS")
print("=" * 80)

# Identify topics with non-zero distribution
assigned_topics = np.where(topic_distribution[doc_id] > 0)[0]

# Sort topics by distribution from highest to lowest
assigned_topics = assigned_topics[
    np.argsort(topic_distribution[doc_id][assigned_topics])[::-1]
]

for topic_id in assigned_topics:
    print(
        f"\nTopic {topic_id}: "
        f"distribution={topic_distribution[doc_id][topic_id]:.3f}, "
        f"relevance={topic_relevance[doc_id][topic_id]:.3f}"
    )

    if topic_id in topic_word_lookup:
        print(
            "Keywords:",
            ", ".join(topic_word_lookup[topic_id][:10])
        )

DOCUMENT 127

Original 20 Newsgroups category:
sci.electronics

Original document:
--------------------------------------------------------------------------------
From: Wayne Alan Martin <wm1h+@andrew.cmu.edu>
Subject: Re: Dayton Hamfest
Organization: Senior, Electrical and Computer Engineering, Carnegie Mellon, Pittsburgh, PA
Lines: 5
Distribution: usa
NNTP-Posting-Host: po5.andrew.cmu.edu
In-Reply-To: <1993Apr19.163122.20454@cbfsb.cb.att.com>

Yes, it is the 23, 24 and 25, in but does anyone have directions how to
get there after I get to Dayton.  Thanks

Wayne Martin



ASSIGNED TOPICS

Topic 158: distribution=0.602, relevance=0.311
Keywords: radio, atari, mhz, contact, hardware, rd, sold, available, interested, disk

Topic 160: distribution=0.125, relevance=0.269
Keywords: hockey, nhl, baseball, batting, clock, ahl, toronto, mcgill, playoff, sport

Topic 126: distribution=0.078, relevance=0.356
Keywords: computing, neural, decode, algorithm, netcom, cs, program, code, software, fr

## 13. Preliminary Qualitative Findings

The feasibility experiment confirms that the reproduced Contextual-Top2Vec implementation is capable of generating multiple topic assignments within individual documents. Across the 200-document sample, documents received between one and eight non-zero topic assignments, with an average of 1.10 topics per document.

However, qualitative inspection indicates that multiple topic assignments should not necessarily be interpreted as accurate within-document segmentation. Document 127, a short post concerning directions to the Dayton Hamfest, received eight topic assignments. The dominant topic contained terms associated with radio and computer hardware, which are reasonably related to the context of a hamfest. Several additional topics also contained electronics and communications-related terms. However, other assigned topics included unrelated terms associated with sports and machine learning or computing.

The corpus-level results show a similar pattern of fragmentation. The model identified 163 topics from only 200 documents, while several of the highest-ranked topics contained strongly overlapping vocabulary. These observations suggest that the reduced corpus and modified clustering thresholds produce an excessively granular topic structure.

Accordingly, the 200-document experiment is treated as a computational feasibility assessment rather than a direct reproduction of the quantitative results reported by Angelov and Inkpen (2024). The experiment nevertheless verifies that the principal components of the Contextual-Top2Vec pipeline, including contextual embedding, topic discovery, document-level topic distributions, relevance scoring, and multiple topic assignment, are operational in the local environment.